# Objetivo 2 (Queries).

Importamos las librerías necesarias.

In [160]:
!pip install -r ../requirements.txt --quiet

In [214]:
import redis
import pandas as pd
from tqdm import tqdm
from pprint import pprint

Nos conectamos a la base de datos.

In [162]:
r = redis.Redis(decode_responses=True)

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 1</h2>

# Diseño del índice.

## Creación de un schema.

### Diseñamos el schema.

El schema schema define principalmente 2 cosas.

- Que campos son los "buscables".
- Y que tipo de busquedas podemos hacer en cada campo.

para elaborarlo primero vamos a intentar comprender el dataset que tenemos. Por lo que que cargamos el dataframe.

In [163]:
df = pd.read_csv("../data/cards_final_with_xp.csv")
df.head(1)

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic


El ejercicio solamente nos pide 3 campos buscables.

- `faction_code`:       De tipo **TAG**
- `traits`:             De tipo **TAG** (Con la particularidad de que una carta puede tener varias tags deparadas con "|" )
- `xp`:                 De tipo **Numérico**.

In [164]:
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.field import NumericField, TagField

In [165]:
schema = (
    TagField("faction_code", separator="|"),
    TagField("traits", separator="|"),
    NumericField("xp")
)

## Creamos un índice

### Creamos el  índice.

In [166]:
index = IndexDefinition(prefix=["cards:"], index_type=IndexType.HASH)

### Inicializamos el indice.

Para evitar problemas eliminamos el indice que ya pueda existir.

In [167]:
idx_name = "idx_1" 

try:
    r.ft(idx_name).dropindex()
    print("El indice se ha reiniciado")
except:
    r.ft(idx_name).create_index(fields=schema, definition=index)
    print("El indice no existía, se creó.")


El indice no existía, se creó.


## Queries

In [168]:
from redis.commands.search.query import Query

### Listado facciones y traits.

Como vamos a estar constantemente trabajando con listados y atributos vamos a realizar un listado de ambos campos.

In [169]:
def get_unique_list(field : str):
    tuples = [trait.split("|") for trait in df[field].unique()]
    list_uniques = set()

    for tuple in tuples:
        for trait in tuple:
            list_uniques.add(trait)

    return list(list_uniques)

traits = get_unique_list("traits")
factions = get_unique_list("faction_code")

print(f"TRAITS: {traits}")
print(f"FACCIONES: {factions}")

TRAITS: ['Civic', 'Instrument', 'Glyph', 'Composure', 'Talent', 'Return', 'Science', 'Desperate', 'Scheme', 'Coastal', 'Central', 'Ship', 'Mainland', 'Cultist', 'Lair', 'Eztli', 'Dhole', 'Ruined', 'Serpent', 'Colour', 'Summit', 'Hemlock Vale', 'Mutated', 'Sanctum', 'Sorcerer', 'Lift', 'Deep One', 'Seafloor', 'Completed', 'Task', 'Innsmouth', 'Act 2', 'Part 1', 'Syndicate', 'Farm', 'Expedition', 'Artifact', 'Apiary', 'Evidence', 'Falcon Point', 'Tarot', 'Glacier', 'Criminal', 'Government', 'Lit', 'Restricted', 'Tindalos', 'Field', 'Pact', 'Oriab', 'Lead', 'Blight', '???', 'Lantern Club', 'Ancient One', 'Cairo', 'Depths', 'Police', 'Unlit', 'Rival', 'Providence', 'Crew', 'Emissary', 'Woods', 'Set', 'Veteran', 'Dark Young', 'Scientist', 'Hazard', 'Unpracticed', 'Cosmos', 'Curse', 'Coterie', 'Resident', 'Leng', 'Footwear', 'Star Spawn', 'Future', 'Risen', 'Endtimes', 'Satellite', 'Blessed', 'Steps', 'Dionsaur', 'Bayou', 'Broken', 'Innate', 'Possessed', 'Corruption', 'Ancient', 'Lodge', 'Re

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 2</h2>

### Query 1:

Dada una lista de facciones devolver cartas que tengan al menos una de las facciones con paginación de 5.

In [248]:
def q1(factions: list[str], page=0):
    factions_str = "|".join(factions)
    q = Query(f"@faction_code:{{{factions_str}}}").paging(offset = page * 5, num = 5)
    search_results = r.ft(idx_name).search(q)
    return search_results

Test de la query 1.

In [253]:
print(q1(["survivor", "guardian"]))
print(q1(["survivor"]))
print(q1(["guardian"]))

Result{493 total, docs: [Document {'id': 'cards:1018', 'payload': None, 'name': 'Beat Cop', 'code': '1018', 'pack_code': 'core', 'illustrator': 'Nicholas Elias', 'image_url': 'https://arkhamdb.com/bundles/cards/01018.png', 'faction_code': 'guardian', 'traits': 'Ally|Police', 'xp': '0', 'type_code': 'asset', 'text': 'You get +1 [combat]. [fast] Discard Beat Cop: Deal 1 damage to an enemy at your location.'}, Document {'id': 'cards:50002', 'payload': None, 'name': 'Dynamite Blast', 'code': '50002', 'pack_code': 'rtnotz', 'illustrator': 'Dimitri Bielak', 'image_url': 'https://arkhamdb.com/bundles/cards/50002.jpg', 'faction_code': 'guardian', 'traits': 'Tactic', 'xp': '2', 'type_code': 'event', 'text': 'Choose either your location or a connecting location. Deal 3 damage to each enemy and to each investigator at the chosen location. This action does not provoke attacks of opportunity.'}, Document {'id': 'cards:60529', 'payload': None, 'name': 'Chainsaw', 'code': '60529', 'pack_code': 'ste',

### Query 2:
Dada una facción cuales son las traits más comunes (agrupadas por cuenta), y ordenadas el orden descendente, paginado a 15 resultados.

Pare resolver esta query usaremos agregaiones, aplicamos un filtro para quedarnos solo con las cartas de la facción dada, luego agrupamos por traits y contamos el número de cartas que tienen cada trait, ordenamos por ese conteo y limitamos a 15 resultados. El detalle esta en usar apply para crear un nuevo campo single_trait que sería la explosión del campo traits que inicialmente es un string con varias traits separadas por "|".

#### Definimos una función para parsear el resultado de la agregación.

Una agregación devuelve una lista de listas por lo que perdemos la asociación clave-valor que tiene la estructura de nuestra carta. Definimos una función para parsear el resultado de la agregación y devolver un listado de diccionarios con la asociación clave-valor, con la opción de pasar un listado de campos para excluir en caso de que no queramos que se devuelvan ciertos campos.

In [206]:
def parse_aggregation_result(result, exclude_fields: list[str] = []):
    parsed_result = []
    
    for row in result:
        pased_row = {}
        i = 0
        while i < len(row):
            key = row[i]
            value = row[i + 1]
            if key not in exclude_fields:
                pased_row[key] = value
                
            i += 2
        parsed_result.append(pased_row)
    return parsed_result

#### Definimos la query 2.

In [250]:
from redis.commands.search.aggregation import AggregateRequest, Desc
from redis.commands.search.reducers import count

def q2(faction: str, page=0):

    req = (AggregateRequest(f"@faction_code:{{{faction}}}") #filtro por facción dada
    .load("@traits") #cargamos el campo traits para poder usarlo en el apply
    .apply(single_trait="split(@traits, '|')") #creamos un nuevo campo single_trait que es la explosión del campo traits
    .group_by("@single_trait", count().alias("total")) #agrupamos por single_trait y contamos el número de cartas que tienen cada trait
    .sort_by(Desc("@total")) #ordenamos por el conteo de cartas que tienen cada trait en orden descendente
    .limit(page * 15, 15)) #limitamos a 15 resultados

    result = r.ft(idx_name).aggregate(req)
    return result.rows

Test query 2.

comprobamos que la query funciona y que también podemos parsear el resultado para obtener un diccionario.

In [251]:
result = q2("survivor")
print(result, end="\n\n")
parsed_result = parse_aggregation_result(result)
pprint(parsed_result)

[['single_trait', 'Item', 'total', '59'], ['single_trait', 'Fortune', 'total', '29'], ['single_trait', 'Ally', 'total', '25'], ['single_trait', 'Blessed', 'total', '25'], ['single_trait', 'Tool', 'total', '22'], ['single_trait', 'Innate', 'total', '22'], ['single_trait', 'Talent', 'total', '21'], ['single_trait', 'Spirit', 'total', '18'], ['single_trait', 'Weapon', 'total', '17'], ['single_trait', 'Melee', 'total', '16'], ['single_trait', 'Tactic', 'total', '16'], ['single_trait', 'Charm', 'total', '14'], ['single_trait', 'Cursed', 'total', '13'], ['single_trait', 'Trick', 'total', '13'], ['single_trait', 'Developed', 'total', '8']]

[{'single_trait': 'Item', 'total': '59'},
 {'single_trait': 'Fortune', 'total': '29'},
 {'single_trait': 'Ally', 'total': '25'},
 {'single_trait': 'Blessed', 'total': '25'},
 {'single_trait': 'Tool', 'total': '22'},
 {'single_trait': 'Innate', 'total': '22'},
 {'single_trait': 'Talent', 'total': '21'},
 {'single_trait': 'Spirit', 'total': '18'},
 {'single_

### Query 3:

 Dado un listado de traits, una facción, y un parámetro exp (opcional). Se devolverán las cartas que contengan todos los traits, priorizando aquellas que tengan la facción dada, y excluyendo aquellas que tengan facción "mythos". Si se ha dado el parámetro exp, se devolverán solo aquellas cartas que tengan un valor de xp menor o igual al dado.

In [242]:
def q3(traits: list[str], faction: str, xp: int = None):

    if xp is None:
        xp_str = "inf"
    else:
        xp_str = str(xp)

    query_list = [f"@xp:[0 {xp_str}]", "-@faction_code:{mythos}"]

    for trait in traits:
        query_list.append(f"@traits:{{{trait}}}")

    req = AggregateRequest(" ".join(query_list)).load()



    req = (
        req.apply(is_trait=f"@faction_code == '{faction}'")
        .sort_by(Desc("@is_trait"))
        .limit(0, 15)
    )

    return r.ft(idx_name).aggregate(req).rows


Test de la query 3.

In [247]:
result = q3(traits=["Clairvoyant"], faction="mystic")
print(result, end="\n\n")

parsed_result = parse_aggregation_result(result)
pprint(parsed_result)


[['faction_code', 'mystic', 'is_trait', '1', 'name', 'Katarina Sojka', 'code', '11069', 'pack_code', 'tdcp', 'illustrator', 'Daniel Watson', 'image_url', 'https://arkhamdb.com/bundles/cards/11069.jpg', 'traits', 'Ally|Clairvoyant|Dreamer', 'xp', '0', 'type_code', 'asset', 'text', '[reaction] When you would reveal chaos tokens during a skill test you are performing, exhaust Katarina Sojka: Reveal tokens from the chaos bag until a token with a symbol is revealed. Resolve that token and ignore the rest.'], ['faction_code', 'mystic', 'is_trait', '1', 'name', 'Jacqueline Fine', 'code', '60401', 'pack_code', 'jac', 'illustrator', 'Aleksander Karcz', 'image_url', 'https://arkhamdb.com/bundles/cards/60401.jpg', 'traits', 'Clairvoyant', 'xp', '0', 'type_code', 'investigator', 'text', '[reaction] When an investigator at your location would reveal any number of chaos tokens: Reveal 2 additional tokens. Of the revealed tokens, choose and cancel 2 non-[auto_fail] tokens, or 1 [auto_fail] token. (Li

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 3</h2>

Durante la implementación ya hemos hecho algunos test individuales. Sin embargo a la hora de asegurarnos de que las queries funcionan correctamente, podemos hacer varias cosas.

- Test genéricos para cada Query. (Lo que ya hemos hecho).
- Test de **edge-cases** como los que detallamos a continuación.
    - Para la query 1: Probar una facción que no exista, o una combinación de facciones donde una no exista.
    - Para la query 2: Probar una facción que no exista, usar una una facción que tenga traits tanto en lista con el caracter "|" como sin el, para asegurarnos de que el apply funciona correctamente.
    - Para la query 3: Probar con traits o facciones que no existan. Comprobar que funciona tanto sin incluir el parámetro exp como incluyéndolo, y que este funciona correctamente. Comprobar que se excluyen las cartas con facción "mythos" aunque cumplan el resto de condiciones.
- Creación de test atomzatizados: También se pueden crear tests automatizados y comprobando los resultados con queries similares definidas en pandsa sobre el dataframe original.
- Creación de una interfaz de usuario: Una interfaz donde puedas seleccionar los parámetros de las queries y mostrar los resultados de forma visual también puede ser una buena forma de testear que todo funciona correctamente.